In [1]:
import polars as pl
import torch


players_df = pl.read_csv('csv/player.csv')

all_ids = players_df['id'].unique()
null_player_id = 0
nba_id_to_idx_mapping = {}
for idx, nba_id in enumerate(all_ids):
    nba_id_to_idx_mapping[nba_id] = idx + 1 # reserve 0 for null player id

nba_id_to_idx_mapping[null_player_id] = 0

max_nba_id = max(nba_id_to_idx_mapping.keys())
lookup_tensor = torch.zeros(max_nba_id + 1, dtype=torch.long)
for nba_id, sequential_idx in nba_id_to_idx_mapping.items():
    lookup_tensor[nba_id] = sequential_idx
    

In [11]:
from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader

REPO_ID = 'sviridov/nba-posession-processed'
processed_dataset = load_dataset(REPO_ID, split='train')

processed_dataset.set_format("torch")

loader = DataLoader(
    processed_dataset, 
    batch_size=2048,      # Critical for GPU utilization
    shuffle=True, 
    num_workers=4,         
    pin_memory=True,       
    persistent_workers=True # Keeps data loading alive between epochs
)


print(len(loader) * 2048, " training shot cycles")

5349376  training shot cycles


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from NBADeepFm import NBATransformer

model = NBATransformer(num_players=len(nba_id_to_idx_mapping), embed_dim=32)
model = model.to(device)


In [ ]:
# train the model
from torch.amp import autocast, GradScaler
from torch import nn

scaler = GradScaler()
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

num_epochs = 1
batch_size = 64

for epoch in range(num_epochs):
    for i, batch in enumerate(loader):
        optimizer.zero_grad()
        inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        labels = inputs.pop("labels")

        with autocast(device_type=device):
            logits = model(**inputs) 
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(loader)}], Loss: {loss.item():.4f}')
    
    torch.save(model, f"nba_transformer_epoch{epoch}_checkpoint.pth")
            


print("Finished training model")

In [ ]:
#upload to hugging face hub


# model.push_to_hub("sviridov/nba-transformer", private=True)